# 🛡️ BorderWatch · Data Analysis Exercises

### Practice Notebook - Solve It Yourself

This notebook contains **20 exercises** that walk you through the full BorderWatch analysis pipeline. Every exercise has:
- A **question** with task description
- An **empty code cell** with comments to guide implementation
- **No solutions** - solve them yourself

The exercises mirror the structure of `Mexico_Cartel_DataSet_v10_3.ipynb` (the analyzer notebook). After finishing all 20, you'll have re-implemented the core of the analyzer.

#### 📐 Exercise Structure

| Part | Topic | # Exercises |
|:---:|:---:|:---:|
| **A** | Data Exploration | 5 |
| **B** | Burner Detection | 4 |
| **C** | Network Analysis | 3 |
| **D** | Statistical Tests | 3 |
| **E** | Composite Score & Ranking | 5 |

#### 📦 Requirements

This notebook assumes the 5 CSVs (or parquet files) from the generator are in the same folder:

```
./manufacturers.csv
./sms.csv
./email.csv
./chat.csv
./waze.csv
./burner_to_owner_ground_truth.csv
```

---


## 🔧 Setup - Load Data

Implement the loading logic. Read all 5 CSV files into pandas DataFrames called `mfg`, `sms`, `eml`, `cht`, `waze`. Also load the ground truth.

For `mfg`, add derived columns: `hour`, `day_of_week`, `is_night` (21:00-04:59), `is_weekend`.

**Goal**: be ready to query each layer.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

np.random.seed(42)




print('mfg :', mfg.shape    if 'mfg' in dir() else '(not loaded)')
print('sms :', sms.shape    if 'sms' in dir() else '(not loaded)')
print('eml :', eml.shape    if 'eml' in dir() else '(not loaded)')
print('cht :', cht.shape    if 'cht' in dir() else '(not loaded)')
print('waze:', waze.shape   if 'waze' in dir() else '(not loaded)')


---

# 📘 Part A - Data Exploration

Get familiar with what you have before doing detection.


## Exercise 1 - Layer Sizes & Time Spans

For each of the 5 layers, compute:
1. Total number of rows
2. Earliest event timestamp
3. Latest event timestamp
4. Total period in days
5. Number of unique senders

**Output**: a single DataFrame `layer_summary` with 5 rows (one per layer) and 5 columns.


In [ ]:







print(layer_summary if 'layer_summary' in dir() else 'Build layer_summary above')


## Exercise 2 - Partner Count Distribution (L1)

For each `caller_phone` in `mfg`, compute the number of unique `receiver_phone` values they contacted.

**Tasks:**
1. Build `partner_counts` series indexed by `caller_phone`, value = `unique_partners`
2. Plot the distribution: histogram with bins 1-25
3. Identify the bin with the largest count - what does that represent?
4. Compute % of phones with exactly 1 partner (burner candidates)

**Hint**: use `mfg.groupby('caller_phone')['receiver_phone'].nunique()`.


## Exercise 3 - Hour-of-Day Distribution Comparison

Compare the hour-of-day distribution for two groups:
- **Group A**: callers with exactly 1 partner (`unique_partners == 1`)
- **Group B**: callers with many partners (`unique_partners >= 5`)

**Tasks:**
1. Compute mean `is_night` rate for each group
2. Plot two histograms (24 bins, hours 0-23) side by side
3. Compute the night ratio difference between groups

What does a higher night ratio for Group A suggest?


## Exercise 4 - Cross-Layer Volume Comparison

For each unique caller_phone in L1 (`mfg`), check whether they also appear in L2 (`sms`) and/or L3 (linked via SIM).

**Tasks:**
1. Compute the set of phones present in L1
2. Compute the set of phones present in L2 (as `sender_phone`)
3. Compute the intersection and difference sets
4. Print a 2×2 contingency table: in-L1 × in-L2

Interpret: are most burners in L1 also active in L2?


## Exercise 5 - City Activity Heatmap

For each city in L1, compute:
1. Total number of calls
2. Number of unique phones
3. Average calls per phone (volume intensity)

**Tasks:**
1. Build `city_stats` DataFrame
2. Sort by total calls descending
3. Plot a horizontal bar chart of the top 12 cities

Which cities are border cities? Which are transit hubs?


---

# 📗 Part B - Burner Detection

Apply detection rules to identify suspicious phones.


## Exercise 6 - Sacred Chart #1: 1:1 Burner Spike

Plot the partner-count distribution **for each of the 4 communication layers** in a 2×2 grid:

| Layer | Source col | Destination col |
|---|---|---|
| L1 GSM | `caller_phone` | `receiver_phone` |
| L2 SMS | `sender_phone` | `receiver_phone` |
| L3 Email | `sender_email` | `receiver_email` |
| L4 Chat | `handler_phone` | `associate_phone` |

**Tasks:**
1. For each layer, group by source and count unique destinations
2. Plot histogram of partner counts (bins 1-15)
3. **Highlight bin=1 in red** for each layer - this is the 1:1 burner spike
4. In how many layers do you see a dominant 1:1 spike?

This chart is the **Sacred Chart #1** of BorderWatch - verify it visually.


## Exercise 7 - Recurring 1:1 Detection (Confirmed Burners)

A "confirmed burner" is a phone that:
1. Has exactly 1 unique partner (1:1 candidate from Ex. 6)
2. Made ≥30 calls to that partner
3. Spans ≥2 days

**Tasks:**
1. For each L1 caller_phone with `unique_partners == 1`, find their single partner
2. Count the total calls and time span (max - min `call_time`) per such pair
3. Filter to keep pairs with `n_calls >= 30` AND `span_days >= 2`
4. The resulting set is `confirmed_l1` - the confirmed burners in L1
5. Print `len(confirmed_l1)` - should be in the hundreds

**Hint**: use `groupby(['caller_phone','receiver_phone'])` on the 1:1 subset.


## Exercise 8 - Pre-Border Blackout

A "pre-border blackout" is a phone that:
1. Has high call volume in border cities (Tijuana, Ciudad Juarez, etc.)
2. Goes silent in the hours before a confirmed border crossing

**Tasks:**
1. For each caller, count calls in any border city
2. Filter: only phones with ≥40 border-city calls
3. Compute the time of their last call before midnight (proxy for "going dark")
4. Rank by border-call volume

What's the top suspect's volume? What's their last-call time?

**Border cities**: `{'Tijuana','Ciudad Juarez','Nogales','Nuevo Laredo','Reynosa','Matamoros','Mexicali','Piedras Negras','Agua Prieta','Sonoyta','Palomas'}`


## Exercise 9 - SIM-Swap Detection

A SIM-swap phone has multiple `caller_sim_id` values associated with the same `caller_phone`.

**Tasks:**
1. For each caller_phone, count unique `caller_sim_id` values
2. Filter phones with `unique_sims >= 2` - these are the swap candidates
3. For each swap phone, also record the count of swaps and the call volume
4. Sort by swap_count descending

How many swap phones do you find? What's the maximum swap count?


---

# 📕 Part C - Network Analysis

Build a graph and find central actors.


## Exercise 10 - Build the Communication Graph

Build a `networkx` graph where:
- Nodes = unique phone numbers (limited to confirmed burners for performance)
- Edges = "called within the period", weighted by call count

**Tasks:**
1. Filter `mfg` to rows where both `caller_phone` and `receiver_phone` are confirmed burners
2. Group by (caller, receiver), get call counts
3. Build a directed `nx.DiGraph` with weighted edges
4. Print: number of nodes, edges, density

**Hint**: `import networkx as nx`; use `G.add_weighted_edges_from(...)`.


## Exercise 11 - Community Detection (Louvain)

Apply the Louvain algorithm to find communities in your graph.

**Tasks:**
1. Convert directed graph to undirected for community detection
2. Run `nx.community.louvain_communities()` or equivalent
3. For each community, count members
4. Sort by community size descending
5. Plot top 12 community sizes (these should match the 12 cartel cells)

How many communities do you find? Is the count close to 12?


## Exercise 12 - Eigenvector Centrality (Top 5 Cell Leaders)

Compute eigenvector centrality on the graph. The top-5 phones with the highest centrality are the cell leader candidates.

**Tasks:**
1. Compute `nx.eigenvector_centrality(G_undirected)`
2. Sort by centrality descending
3. Print the top 10 phones with their centrality score
4. For the top 5, look up their attributes from `mfg` (city, total calls, partners)

Compare to the ground truth: do your top-5 match real commander phones?


---

# 📊 Part D - Statistical Tests

Validate detection with statistical evidence.


## Exercise 13 - Test A: Night Activity (Chi²)

Hypothesis: burner phones operate disproportionately at night.

**Tasks:**
1. For each `caller_phone`, compute night_ratio = mean(`is_night`)
2. Split into 2 groups: confirmed burners vs. all other phones
3. Compute mean night ratio for each group
4. Run a chi² test on a 2×2 contingency table (night vs. day × burner vs. legit)
5. Compute Cramér's V as the effect size

Expected: burner night ratio ~65%, legit ~12%, Cramér's V ≈ 0.38 (medium).


## Exercise 14 - Test B: Call Duration (Mann-Whitney U)

Hypothesis: burner calls have a different median duration than legitimate calls.

**Tasks:**
1. Get `call_duration_sec` for all rows where caller is a confirmed burner
2. Get `call_duration_sec` for all rows where caller is NOT a confirmed burner
3. Compute median for each group
4. Run `scipy.stats.mannwhitneyu(burner_durations, legit_durations)`
5. Compute the rank-biserial r effect size
6. Direction: do burners call **shorter** or **longer**? Print accordingly.

Expected: burner median ~107s, legit median ~330s, r ≈ -0.29 (small), direction = SHORTER.


## Exercise 15 - Test C: SIM-Missing Rate (Odds Ratio)

Hypothesis: burner phones have a higher rate of missing/null SIM values (counter-surveillance signal).

**Tasks:**
1. For each call, flag `sim_missing = (caller_sim_id is null)`
2. Split into 2 groups: confirmed burners vs legit
3. Compute null rate per group
4. Build a 2×2 contingency table (null vs not-null × burner vs legit)
5. Compute the Odds Ratio (OR) and its 95% CI
6. Interpret: how many times more likely are burners to have null SIM?

Expected: burner null rate ~10%, legit ~1%, OR ~9× (Large).


---

# 🎯 Part E - Composite Score & Ranking

Combine everything into a final suspect list.


## Exercise 16 - Build Per-Phone Signal Matrix

For every phone in the dataset, build a binary signal vector with 7 signals:

| Signal | Meaning |
|---|---|
| `is_burner` | 1 if confirmed burner (Ex. 7) |
| `is_op` | 1 if total_calls > 85th percentile |
| `has_chain` | 1 if found in cross-layer chain (L1↔L2↔L3) |
| `null_sim` | 1 if null_sim_rate > 0.10 |
| `sim_swap` | 1 if sim_swap_count >= 2 |
| `blackout` | 1 if in pre-border blackout list (Ex. 8) |
| `is_night` | 1 if night_ratio > 0.50 |

**Tasks:**
1. Build a DataFrame `signals` with index = caller_phone and 7 binary columns
2. Print head() and basic statistics


## Exercise 17 - Composite Score (Weighted Dot Product)

Compute a composite suspicion score using the weights:

$$w = [30, 15, 20, 10, 10, 10, 5]$$

The maximum possible score is 100.

**Tasks:**
1. Multiply the signal matrix by `w` (matrix multiplication or `@` operator)
2. Add `composite_score` column to your signals DataFrame
3. Compute the distribution: how many phones at score≥75 (CRITICAL), 50-74 (HIGH), <50 (LOW)
4. Plot the distribution as a histogram with the thresholds marked


## Exercise 18 - Identity Attribution (Cell-Tower Co-Location)

For each confirmed burner, find the dominant non-burner phone at the same cell tower.

**Tasks:**
1. For each confirmed burner, find their most-common `cell_tower_id` in `mfg`
2. Filter `mfg` to non-burner rows at that tower
3. Find the non-burner caller_phone with the most calls at that tower → this is the attributed identity
4. Compute confidence_pct = (burner's calls at tower) / (non-burner's total calls) × 100
5. Build the attribution table

What's the mean attribution confidence?


## Exercise 19 - Top 20 Priority Suspects Table

Build the executive deliverable: the top 20 suspects with all relevant fields.

**Tasks:**
1. Join: signals + composite_score + attribution + activity stats
2. Sort by composite_score descending
3. Take the top 20
4. Display columns: `suspect_name`, `composite_score`, `priority_tier`, `evidence_signals`, `legit_phone`, `total_calls`

Format `evidence_signals` as a list of activated signal names (e.g., "is_burner | null_sim | sim_swap").


## Exercise 20 - Synthesis: Save Excel Report

Build a 2-sheet Excel file:

**Sheet 1 - Named Suspects** (~145 rows):
- One row per person (aggregated by suspect_name + legit_phone)
- 12 columns: suspect_name, attribution_score, layer, email, burner_phone (joined by |), legit_phone, activity_volume (sum), contacts_count, contacts_detail, central_score, priority_tier, matched_city

**Sheet 2 - Anonymous Burners** (~1,648 rows):
- One row per burner phone that couldn't be attributed
- 9 columns: burner_phone, layer, email, activity_volume, contacts_count, contacts_detail, central_score, priority_tier, matched_city

**Tasks:**
1. Aggregate by (suspect_name, legit_phone) using groupby
2. For burner_phone - join all burners with `|`
3. For layer - join unique layers with `,`
4. Save to `my_suspects_report.xlsx` using pd.ExcelWriter with `openpyxl`

This replicates the Excel export from Cell 159 of the analyzer notebook.


---

# 🎓 Summary - What You've Built

You've reimplemented the **core pipeline** of `Mexico_Cartel_DataSet_v10_3.ipynb`:

| Exercise | Concept | Maps to |
|:---:|---|---|
| 1-5 | Data exploration | Stage 1 of analyzer |
| 6 | Sacred Chart #1 - 1:1 spike | Cell 28 |
| 7 | Confirmed burner detection | Stage 2 (Cell 49) |
| 8 | Pre-border blackout | Stage 5 (Cell 95) |
| 9 | SIM-swap detection | Stage 5 |
| 10-12 | Network analysis | Stage 4 (Cells 70-92) |
| 13-15 | Tests A/B/C | Stage 6 (Cells 110-111) |
| 16-17 | Composite scoring | Stage 9 (Cell 81) |
| 18 | Identity attribution | Stage 9 (Cell 81) |
| 19-20 | Executive deliverable | Stage 12 (Cell 159) |

#### 🎯 What's Missing (for advanced practice)

If you want to continue, the full analyzer notebook also includes:
- **Random Forest ML pipeline** (Stage 6, Cells 102-120) - train a classifier on the 8 features
- **F2 threshold tuning** (Cell 112) - Precision-Recall curve and confusion matrix viz
- **SHAP feature importance** (Cell 119) - interpret the model
- **Folium interactive map** (Cell 138) - 8-layer map with toggleable views
- **HTML dashboard** (Cell 159) - composite report with embedded charts and Excel download

Compare your implementations to the corresponding cells in the analyzer notebook for reference.

---
